# Advisor Module — Extraction Accuracy Evaluation

Evaluates the Advisor agent's ability to correctly extract or infer structured fields from
the customer conversation, using pre-computed results from `actual_test_cases/advisor_result.xlsx`.

## Fields evaluated

| Field | Type | Metric |
|---|---|---|
| `consultAcc` | String/ID extraction | Null-aware accuracy (normalised) |
| `DebtSituation` | Categorical | Accuracy + Confusion matrix + Precision/Recall |
| `maxPayment` | Numeric extraction | Null-aware exact-match accuracy |
| `maxTerm` | Numeric extraction | Null-aware exact-match accuracy |
| `refPlanID` | String/ID extraction | Null-aware accuracy |
| `reConfirmMessage` | Categorical | Accuracy + Confusion matrix + Precision/Recall |
| `maxPaymentY2` | ⚠️ Skipped | No expected values in test set |
| `maxPaymentY3` | ⚠️ Skipped | No expected values in test set |

### Null-handling convention (applied to all fields)
| Expected | Result | Classification | Correct? |
|---|---|---|---|
| null | null | CORRECT_ABSTENTION | ✓ |
| null | non-null | FALSE_POSITIVE | ✗ |
| non-null | null | FALSE_NEGATIVE | ✗ |
| non-null | non-null (match) | CORRECT_MATCH | ✓ |
| non-null | non-null (mismatch) | VALUE_MISMATCH | ✗ |

### Metric rationale
- **Accuracy** is used for numeric and ID/string extraction fields because the target is a specific value (not a fixed category set); the relevant question is simply whether the model extracted the right value
- **Confusion matrix + Precision/Recall** is additionally applied to categorical fields (`DebtSituation`, `reConfirmMessage`) because these come from a known finite label set, making per-class recall (sensitivity) and precision (specificity) meaningful — especially important for imbalanced or multi-class distributions.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

print('Libraries loaded.')

## 1. Configuration

In [ ]:
DATA_FILE   = "../actual_test_cases/advisor_result.xlsx"
OUTPUT_FILE = "advisor_eval_results.xlsx"

# Fields to evaluate and their type
VALUE_FIELDS       = ["consultAcc", "maxPayment", "maxTerm", "refPlanID"]   # accuracy metric
CATEGORICAL_FIELDS = ["DebtSituation", "reConfirmMessage"]                  # confusion matrix + P/R
SKIP_FIELDS        = ["maxPaymentY2", "maxPaymentY3"]                       # no expected labels
ALL_EVAL_FIELDS    = VALUE_FIELDS + CATEGORICAL_FIELDS

# Null sentinel used in categorical confusion matrices
NONE_LABEL = "NONE"

print('Config ready.')

## 2. Load Data

In [ ]:
df = pd.read_excel(DATA_FILE)
print(f"Loaded {len(df)} rows × {len(df.columns)} columns")
print("Columns:", df.columns.tolist())
print(f"\nErrors: {df['error'].notna().sum()}")
df.head(3)

## 3. Field Distributions

In [ ]:
print("=" * 65)
print("FIELD DISTRIBUTIONS  (expected vs result)")
print("=" * 65)

for f in ALL_EVAL_FIELDS:
    exp = df[f"expected_{f}"]
    res = df[f"result_{f}"]
    print(f"\n--- {f} ---")
    print(f"  dtype            : expected={exp.dtype}  result={res.dtype}")
    print(f"  non-null (expected): {exp.notna().sum()}/{len(df)}")
    print(f"  non-null (result)  : {res.notna().sum()}/{len(df)}")
    if pd.api.types.is_object_dtype(exp):
        print(f"  expected values : {sorted(exp.dropna().unique(), key=str)}")
        print(f"  result values   : {sorted(res.dropna().unique(), key=str)}")
    else:
        if exp.notna().any():
            print(f"  expected range  : {exp.min()} – {exp.max()}")
        if res.notna().any():
            print(f"  result range    : {res.min()} – {res.max()}")

print(f"\n--- maxPaymentY2 / maxPaymentY3 (SKIPPED) ---")
for f in SKIP_FIELDS:
    print(f"  {f}: expected non-null={df[f'expected_{f}'].notna().sum()}, "
          f"result non-null={df[f'result_{f}'].notna().sum()} "
          f"(result values: {df[f'result_{f}'].dropna().unique().tolist()[:5]})")

## 4. Compute Match Columns

For each evaluated field we add:
- `null_status_<field>` — five-way classification of each row's null/value pattern.
- `match_<field>` — `True` = correct (CORRECT_ABSTENTION or CORRECT_MATCH), `False` = any error.

In [ ]:
def normalise_acc(s):
    """Sort and strip account-number lists so spacing/order differences don't cause mismatches."""
    if pd.isna(s):
        return None
    parts = [x.strip() for x in str(s).replace(' ', '').split(',')]
    return ','.join(sorted(parts))


def null_status(exp, res, normalise_fn=None):
    if pd.isna(exp) and pd.isna(res):
        return "CORRECT_ABSTENTION"
    if pd.isna(exp) and not pd.isna(res):
        return "FALSE_POSITIVE"
    if not pd.isna(exp) and pd.isna(res):
        return "FALSE_NEGATIVE"
    # both present
    e = normalise_fn(exp) if normalise_fn else str(exp).strip()
    r = normalise_fn(res) if normalise_fn else str(res).strip()
    return "CORRECT_MATCH" if e == r else "VALUE_MISMATCH"


CORRECT_STATUSES = {"CORRECT_ABSTENTION", "CORRECT_MATCH"}

normalise_fns = {
    "consultAcc": normalise_acc,
    "maxPayment": None,
    "maxTerm":    None,
    "refPlanID":  None,
    "DebtSituation": None,
    "reConfirmMessage": None,
}

df_out = df.copy()

for f in ALL_EVAL_FIELDS:
    fn = normalise_fns.get(f)
    statuses = [
        null_status(row[f"expected_{f}"], row[f"result_{f}"], fn)
        for _, row in df.iterrows()
    ]
    df_out[f"null_status_{f}"] = statuses
    df_out[f"match_{f}"]       = [s in CORRECT_STATUSES for s in statuses]

# Row-level: all evaluated fields correct
df_out["row_all_match"] = df_out[[f"match_{f}" for f in ALL_EVAL_FIELDS]].all(axis=1)

print("Match columns added.")
print("Row-level accuracy (all fields correct):",
      f"{df_out['row_all_match'].sum()}/{len(df_out)} "
      f"({df_out['row_all_match'].mean():.1%})")

## 5. Accuracy — Value & Numeric Extraction Fields

Fields: `consultAcc`, `maxPayment`, `maxTerm`, `refPlanID`

Metric: **null-aware exact-match accuracy** over all 250 rows.

In [ ]:
STATUS_ORDER = ["CORRECT_ABSTENTION", "CORRECT_MATCH", "VALUE_MISMATCH", "FALSE_POSITIVE", "FALSE_NEGATIVE"]

accuracy_rows = []
print("=" * 65)
print("VALUE / NUMERIC EXTRACTION — accuracy per field")
print("=" * 65)

for f in VALUE_FIELDS:
    counts = df_out[f"null_status_{f}"].value_counts()
    total  = len(df_out)
    correct = df_out[f"match_{f}"].sum()
    acc    = correct / total
    row = {"field": f, "total": total, "correct": correct, "accuracy": round(acc, 4)}
    for s in STATUS_ORDER:
        row[s] = int(counts.get(s, 0))
    accuracy_rows.append(row)

    print(f"\n  {f}")
    print(f"    Accuracy : {acc:.1%}  ({correct}/{total})")
    for s in STATUS_ORDER:
        n = int(counts.get(s, 0))
        if n:
            print(f"    {s:25s}: {n}")

df_accuracy = pd.DataFrame(accuracy_rows)
display(df_accuracy)

In [ ]:
# Bar chart — accuracy per value field
fig, axes = plt.subplots(1, len(VALUE_FIELDS), figsize=(14, 4))
fig.suptitle("Advisor — Value/Numeric Extraction: Null-Aware Status Breakdown", fontweight="bold")

STATUS_COLORS = {
    "CORRECT_ABSTENTION": "#4caf50",
    "CORRECT_MATCH":      "#8bc34a",
    "VALUE_MISMATCH":     "#ff9800",
    "FALSE_POSITIVE":     "#f44336",
    "FALSE_NEGATIVE":     "#9c27b0",
}

for ax, f in zip(axes, VALUE_FIELDS):
    counts = df_out[f"null_status_{f}"].value_counts().reindex(STATUS_ORDER, fill_value=0)
    colors = [STATUS_COLORS[s] for s in STATUS_ORDER]
    bars = ax.bar(range(len(STATUS_ORDER)), counts.values, color=colors, edgecolor='white')
    ax.set_xticks(range(len(STATUS_ORDER)))
    ax.set_xticklabels([s.replace('_', '\n') for s in STATUS_ORDER], fontsize=7)
    ax.set_title(f, fontweight='bold')
    acc = df_out[f"match_{f}"].mean()
    ax.set_xlabel(f"Accuracy: {acc:.1%}", fontsize=9)
    for bar, val in zip(bars, counts.values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(val),
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylim(0, max(counts.values) * 1.2 + 2)

plt.tight_layout()
plt.savefig("advisor_value_extraction.png", dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved → advisor_value_extraction.png")

## 6. Categorical Fields — Confusion Matrix + Precision/Recall

Fields: `DebtSituation`, `reConfirmMessage`

Null expected/result is mapped to the label `"NONE"` before computing confusion matrices
so that abstentions are treated as a separate class.

In [ ]:
def to_label(s):
    return NONE_LABEL if pd.isna(s) else str(s).strip()


cat_metrics: dict[str, dict] = {}

print("=" * 65)
print("CATEGORICAL FIELDS — per-class metrics")
print("=" * 65)

for f in CATEGORICAL_FIELDS:
    y_true = df_out[f"expected_{f}"].apply(to_label)
    y_pred = df_out[f"result_{f}"].apply(to_label)

    classes = sorted(set(y_true.unique()) | set(y_pred.unique()))
    # put NONE first so it is the 'negative' class
    if NONE_LABEL in classes:
        classes = [NONE_LABEL] + [c for c in classes if c != NONE_LABEL]

    overall_acc = accuracy_score(y_true, y_pred)

    report_dict = classification_report(
        y_true, y_pred, labels=classes, zero_division=0, output_dict=True
    )

    report_rows = []
    for cls in classes:
        r = report_dict.get(cls, {})
        report_rows.append({
            "class": cls,
            "support (expected)": int(y_true.eq(cls).sum()),
            "support (predicted)": int(y_pred.eq(cls).sum()),
            "precision": round(r.get("precision", 0), 4),
            "recall":    round(r.get("recall", 0), 4),
            "f1-score":  round(r.get("f1-score", 0), 4),
        })

    df_report = pd.DataFrame(report_rows)
    cat_metrics[f] = {
        "overall_accuracy": overall_acc,
        "classes": classes,
        "report_df": df_report,
        "y_true": y_true,
        "y_pred": y_pred,
        "cm": confusion_matrix(y_true, y_pred, labels=classes),
    }

    print(f"\n  {f}  |  Overall accuracy: {overall_acc:.1%}")
    display(df_report)
    print()
    print(classification_report(y_true, y_pred, labels=classes, zero_division=0))

In [ ]:
# Confusion matrix heatmaps for categorical fields
fig, axes = plt.subplots(1, len(CATEGORICAL_FIELDS),
                         figsize=(7 * len(CATEGORICAL_FIELDS), 5))
if len(CATEGORICAL_FIELDS) == 1:
    axes = [axes]

fig.suptitle("Advisor — Categorical Fields: Confusion Matrices", fontweight="bold")

for ax, f in zip(axes, CATEGORICAL_FIELDS):
    m = cat_metrics[f]
    cmd = ConfusionMatrixDisplay(
        confusion_matrix=m["cm"],
        display_labels=m["classes"],
    )
    cmd.plot(ax=ax, colorbar=False, cmap="Blues", xticks_rotation="vertical")
    ax.set_title(f"{f}\n(accuracy {m['overall_accuracy']:.1%})", fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Expected")

plt.tight_layout()
plt.savefig("advisor_categorical_cm.png", dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved → advisor_categorical_cm.png")

## 7. Field-Level Summary

In [ ]:
summary_rows = []

for f in VALUE_FIELDS:
    counts = df_out[f"null_status_{f}"].value_counts()
    summary_rows.append({
        "field":              f,
        "type":               "value/numeric extraction",
        "metric":             "null-aware accuracy",
        "total_rows":         len(df_out),
        "correct":            int(df_out[f"match_{f}"].sum()),
        "accuracy":           round(df_out[f"match_{f}"].mean(), 4),
        "CORRECT_ABSTENTION": int(counts.get("CORRECT_ABSTENTION", 0)),
        "CORRECT_MATCH":      int(counts.get("CORRECT_MATCH", 0)),
        "VALUE_MISMATCH":     int(counts.get("VALUE_MISMATCH", 0)),
        "FALSE_POSITIVE":     int(counts.get("FALSE_POSITIVE", 0)),
        "FALSE_NEGATIVE":     int(counts.get("FALSE_NEGATIVE", 0)),
        "precision_macro":    None,
        "recall_macro":       None,
        "f1_macro":           None,
        "notes":              "",
    })

for f in CATEGORICAL_FIELDS:
    m = cat_metrics[f]
    counts = df_out[f"null_status_{f}"].value_counts()
    y_true, y_pred = m["y_true"], m["y_pred"]
    summary_rows.append({
        "field":              f,
        "type":               "categorical extraction",
        "metric":             "accuracy + confusion matrix",
        "total_rows":         len(df_out),
        "correct":            int(df_out[f"match_{f}"].sum()),
        "accuracy":           round(m["overall_accuracy"], 4),
        "CORRECT_ABSTENTION": int(counts.get("CORRECT_ABSTENTION", 0)),
        "CORRECT_MATCH":      int(counts.get("CORRECT_MATCH", 0)),
        "VALUE_MISMATCH":     int(counts.get("VALUE_MISMATCH", 0)),
        "FALSE_POSITIVE":     int(counts.get("FALSE_POSITIVE", 0)),
        "FALSE_NEGATIVE":     int(counts.get("FALSE_NEGATIVE", 0)),
        "precision_macro":    round(precision_score(y_true, y_pred, average='macro', zero_division=0), 4),
        "recall_macro":       round(recall_score(y_true, y_pred, average='macro', zero_division=0), 4),
        "f1_macro":           round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
        "notes":              f"classes: {m['classes']}",
    })

for f in SKIP_FIELDS:
    summary_rows.append({
        "field":              f,
        "type":               "numeric extraction",
        "metric":             "SKIPPED",
        "total_rows":         len(df_out),
        "correct":            None,
        "accuracy":           None,
        "CORRECT_ABSTENTION": None,
        "CORRECT_MATCH":      None,
        "VALUE_MISMATCH":     None,
        "FALSE_POSITIVE":     None,
        "FALSE_NEGATIVE":     None,
        "precision_macro":    None,
        "recall_macro":       None,
        "f1_macro":           None,
        "notes":              "No expected labels in test set; result is always 0.",
    })

df_field_summary = pd.DataFrame(summary_rows)
print("=" * 65)
print("FIELD-LEVEL SUMMARY")
print("=" * 65)
display(df_field_summary)

## 8. Categorical Per-Class Detail Tables

In [ ]:
cat_report_dfs: dict[str, pd.DataFrame] = {}
for f in CATEGORICAL_FIELDS:
    print(f"\n{'=' * 55}")
    print(f"  {f}  — per-class metrics (NONE = absent/not extracted)")
    print(f"{'=' * 55}")
    cat_report_dfs[f] = cat_metrics[f]["report_df"]
    display(cat_metrics[f]["report_df"])

## 9. Mismatch Inspection

In [ ]:
ERROR_STATUSES = {"VALUE_MISMATCH", "FALSE_POSITIVE", "FALSE_NEGATIVE"}
BASE_COLS = ["testId", "userMessage"]

for f in ALL_EVAL_FIELDS:
    bad = df_out[df_out[f"null_status_{f}"].isin(ERROR_STATUSES)]
    if bad.empty:
        continue
    print(f"\n{'=' * 70}")
    print(f"[{f}]  {len(bad)} mismatched rows")
    print(f"{'=' * 70}")
    show_cols = BASE_COLS + [f"expected_{f}", f"result_{f}", f"null_status_{f}"]
    display(bad[show_cols].reset_index(drop=True))

## 10. Export Excel — `overview` + `test_result`

In [ ]:
# ── Build test_result sheet ────────────────────────────────────────────────
# Columns: all original + null_status_*, match_*, row_all_match
test_result_cols = list(df.columns)
for f in ALL_EVAL_FIELDS:
    test_result_cols += [f"null_status_{f}", f"match_{f}"]
test_result_cols.append("row_all_match")

df_test_result = df_out[test_result_cols].copy()

# ── Build overview content ────────────────────────────────────────────────
total_rows    = len(df_out)
row_acc       = df_out["row_all_match"].mean()
row_correct   = int(df_out["row_all_match"].sum())

# Section A: test metadata
df_meta = pd.DataFrame([
    {"metric": "Total test cases",                   "value": total_rows},
    {"metric": "Errors / failed calls",              "value": int(df["error"].notna().sum())},
    {"metric": "Row-level accuracy (all fields)",    "value": f"{row_correct}/{total_rows}  ({row_acc:.1%})"},
    {"metric": "Fields evaluated",                   "value": ", ".join(ALL_EVAL_FIELDS)},
    {"metric": "Fields skipped (no expected labels)","value": ", ".join(SKIP_FIELDS)},
])

# Section B: field-level summary
df_overview_fields = df_field_summary[[
    "field", "type", "metric", "correct", "accuracy",
    "CORRECT_ABSTENTION", "CORRECT_MATCH", "VALUE_MISMATCH",
    "FALSE_POSITIVE", "FALSE_NEGATIVE",
    "precision_macro", "recall_macro", "f1_macro", "notes"
]].copy()

# Section C: per-class detail for categorical fields
cat_class_rows = []
for f in CATEGORICAL_FIELDS:
    for _, row in cat_metrics[f]["report_df"].iterrows():
        cat_class_rows.append({"field": f, **row.to_dict()})
df_cat_detail = pd.DataFrame(cat_class_rows)

# ── Write Excel ────────────────────────────────────────────────────────────
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:

    # ── overview sheet ────────────────────────────────────────────────────
    ws_name = "overview"
    cur_row = 0

    def write_section(title: str, df_section: pd.DataFrame, gap: int = 2) -> int:
        nonlocal cur_row
        # title as single-cell row
        title_df = pd.DataFrame([[title]])
        title_df.to_excel(writer, sheet_name=ws_name, startrow=cur_row,
                          index=False, header=False)
        cur_row += 1
        df_section.to_excel(writer, sheet_name=ws_name, startrow=cur_row,
                            index=False)
        cur_row += len(df_section) + 1 + gap
        return cur_row

    write_section("=== TEST METADATA ===", df_meta)
    write_section("=== FIELD-LEVEL SUMMARY ===", df_overview_fields)
    write_section("=== CATEGORICAL FIELDS — PER-CLASS DETAIL ===", df_cat_detail)

    # ── test_result sheet ─────────────────────────────────────────────────
    df_test_result.to_excel(writer, sheet_name="test_result", index=False)

print(f"Saved → {OUTPUT_FILE}")
print(f"  Sheet 'overview'    : metadata + field-level summary + per-class detail")
print(f"  Sheet 'test_result' : {len(df_test_result)} rows × {len(df_test_result.columns)} columns")

## 11. Final Summary

In [ ]:
print("=" * 65)
print("ADVISOR EVALUATION — FINAL SUMMARY")
print("=" * 65)
print(f"  Total test cases  : {total_rows}")
print(f"  Row-level accuracy: {row_correct}/{total_rows} ({row_acc:.1%})")
print()
print("  Per-field accuracy:")
for f in ALL_EVAL_FIELDS:
    acc  = df_out[f"match_{f}"].mean()
    corr = df_out[f"match_{f}"].sum()
    ftype = "categorical" if f in CATEGORICAL_FIELDS else "value/numeric"
    counts = df_out[f"null_status_{f}"].value_counts()
    vm  = counts.get("VALUE_MISMATCH", 0)
    fp  = counts.get("FALSE_POSITIVE", 0)
    fn  = counts.get("FALSE_NEGATIVE", 0)
    print(f"    {f:22s} [{ftype:20s}]  acc={acc:.1%}  "
          f"VM={vm}  FP={fp}  FN={fn}")
print()
print("  Categorical fields — macro-F1:")
for f in CATEGORICAL_FIELDS:
    m = cat_metrics[f]
    f1  = f1_score(m["y_true"], m["y_pred"], average='macro', zero_division=0)
    rec = recall_score(m["y_true"], m["y_pred"], average='macro', zero_division=0)
    pre = precision_score(m["y_true"], m["y_pred"], average='macro', zero_division=0)
    print(f"    {f:22s}  precision={pre:.1%}  recall={rec:.1%}  f1={f1:.1%}")
print()
print("  Skipped (no expected labels):")
for f in SKIP_FIELDS:
    print(f"    {f}")